# Matemáticas de la Inteligencia Artificial
## Sesión 11 — Embeddings y el primer modelo neuronal de lenguaje

[![Abrir en Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CuentosCuanticos/matematicas-ia/blob/main/11_lm_neuronal/laboratorio.ipynb)

**Pregunta:** ¿cómo pasa un token discreto a una representación aprendida y cómo usa una red una ventana de contexto para predecir el siguiente token?

Recorrido: $\text{token}\to\text{índice}\to\text{embedding}\to\text{ventana}\to\text{MLP}\to\text{logits}\to\text{cross-entropy}\to\text{gradientes}$.

Completa los `TODO` y, sobre todo, explica qué objeto matemático representa cada tensor.

In [ ]:
import math, random, numpy as np, torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
SEED=11
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(torch.__version__, device)

## 1. Vocabulario, índices y one-hot
Un índice es una etiqueta, no una coordenada semántica. Para one-hot, $e_i^Te_j=\delta_{ij}$ y, si $i\ne j$, $\|e_i-e_j\|=\sqrt2$.

In [ ]:
corpus=[
'la luz viaja por el espacio','la materia tiene masa','la masa curva el espacio',
'un modelo aprende patrones','un modelo predice tokens','el lenguaje es una secuencia',
'una red aprende representaciones','un embedding representa un token',
'el contexto cambia la prediccion','la probabilidad mide incertidumbre',
'el gradiente cambia los parametros','una maquina aprende de datos']
BOS='<BOS>'; EOS='<EOS>'
# TODO 1
caracteres=...
vocabulario=...
V=len(vocabulario)
token_a_id=...
id_a_token=...

def codificar(s):
    # TODO 2
    return ...

def decodificar(ids):
    # TODO 3
    return ...

assert ... # reversibilidad para todo el corpus
ids=torch.tensor([token_a_id[t] for t in ['a','e',' ','l']])
O=... # TODO 4: F.one_hot(...).float()
print('Gram=\n', ... ) # TODO 5
print('distancias=\n', torch.cdist(O,O))

## 2. Embedding = lookup entrenable
PyTorch almacena $E\in\mathbb R^{V\times d}$. `nn.Embedding(V,d)` devuelve la fila asociada al índice. Comprueba que lookup y one-hot por matriz son la misma operación.

In [ ]:
d=8
emb=nn.Embedding(V,d)
ids=torch.tensor([token_a_id['a'],token_a_id['e'],token_a_id[' ']])
X_lookup=... # TODO 6
O=F.one_hot(ids,num_classes=V).float()
X_matriz=... # TODO 7: O @ emb.weight
print(X_lookup.shape, (X_lookup-X_matriz).abs().max().item())
assert ...

## 3. Ventanas de contexto
Con longitud $m$, convertimos el texto en ejemplos $(t_{k-m},\ldots,t_{k-1})\to t_k$. El propio texto suministra las etiquetas: es aprendizaje autosupervisado.

In [ ]:
def crear_ejemplos(frases,m):
    X,y=[],[]
    for frase in frases:
        ids=[token_a_id[BOS]]*m+[token_a_id[c] for c in frase]+[token_a_id[EOS]]
        for k in range(m,len(ids)):
            # TODO 8
            X.append(...)
            y.append(...)
    return torch.tensor(X),torch.tensor(y)

m=4
X,y=crear_ejemplos(corpus,m)
perm=torch.randperm(len(X)); cut=int(.8*len(X))
tr,va=perm[:cut],perm[cut:]
Xtr,ytr=X[tr],y[tr]; Xva,yva=X[va],y[va]
print(Xtr.shape,Xva.shape)

## 4. Primer modelo neuronal de lenguaje
Para un batch: $(B,m)\to(B,m,d)\to(B,md)\to(B,h)\to(B,V)$. La salida son **logits**; `cross_entropy` aplica internamente la combinación estable log-softmax + NLL.

In [ ]:
class ModeloVentana(nn.Module):
    def __init__(self,V,d,m,h):
        super().__init__(); self.m=m
        self.embedding=nn.Embedding(V,d)
        self.fc1=nn.Linear(m*d,h)
        self.fc2=nn.Linear(h,V)
    def forward(self,ids):
        x=self.embedding(ids)
        # TODO 9: concatena los m embeddings
        c=...
        h=... # TODO 10: tanh o ReLU
        return ... # TODO 11

modelo=ModeloVentana(V,d=12,m=m,h=48).to(device)
xb=Xtr[:16].to(device); yb=ytr[:16].to(device)
logits=modelo(xb)
print('logits',logits.shape)
loss=F.cross_entropy(logits,yb)
print('CE=',loss.item(),'PPL=',math.exp(loss.item()))

## 5. La identidad clave: $\nabla_z\mathcal L=p-y$
Para un objetivo one-hot $y$ y $p=\mathrm{softmax}(z)$, la cross-entropy satisface $\partial\mathcal L/\partial z_j=p_j-y_j$. Verifícalo con autograd.

In [ ]:
z=torch.tensor([[1.0,1.5,0.0,2.5]],requires_grad=True)
target=torch.tensor([1])
L=F.cross_entropy(z,target); L.backward()
p=F.softmax(z.detach(),dim=1)
yoh=F.one_hot(target,num_classes=4).float()
print('autograd:',z.grad)
print('p-y     :',p-yoh)
assert ... # TODO 12

## 6. Entrenamiento y gradientes del embedding
En descenso de gradiente, $\theta\leftarrow\theta-\eta\nabla_\theta\mathcal L$. Las filas de $E$ que reciben gradiente corresponden a tokens presentes en los contextos del batch.

In [ ]:
def evaluar(modelo,X,y):
    modelo.eval()
    with torch.no_grad():
        z=modelo(X.to(device)); ce=F.cross_entropy(z,y.to(device)).item()
        acc=(z.argmax(1)==y.to(device)).float().mean().item()
    return ce,acc

def entrenar(modelo,X,y,epochs=250,lr=.03):
    opt=torch.optim.AdamW(modelo.parameters(),lr=lr)
    hist=[]
    for ep in range(epochs):
        modelo.train(); opt.zero_grad()
        z=modelo(X.to(device)); loss=F.cross_entropy(z,y.to(device))
        loss.backward(); opt.step(); hist.append(loss.item())
    return hist

hist=... # TODO 13
ce,acc=evaluar(modelo,Xva,yva)
print('val CE=',ce,'PPL=',math.exp(ce),'acc=',acc)
plt.plot(hist); plt.xlabel('epoch'); plt.ylabel('CE train'); plt.show()

modelo.zero_grad(); z=modelo(xb); F.cross_entropy(z,yb).backward()
filas=set(torch.where(modelo.embedding.weight.grad.norm(dim=1)>1e-12)[0].tolist())
presentes=set(xb.detach().cpu().reshape(-1).tolist())
print('filas con gradiente:',len(filas),'tokens presentes:',len(presentes))
assert ... # TODO 14: filas <= presentes

## 7. Generación y geometría aprendida
Generamos autorregresivamente y proyectamos los embeddings con PCA/SVD. No atribuyas significado absoluto a los ejes: la representación se optimiza para la tarea.

In [ ]:
@torch.no_grad()
def generar(modelo,max_pasos=100,T=1.0):
    modelo.eval(); contexto=[token_a_id[BOS]]*modelo.m; out=[]
    for _ in range(max_pasos):
        x=torch.tensor([contexto[-modelo.m:]],device=device)
        z=modelo(x)[0]
        probs=... # TODO 15: softmax(z/T)
        j=int(torch.multinomial(probs,1))
        tok=id_a_token[j]
        if tok==EOS: break
        if tok!=BOS: out.append(tok)
        contexto.append(j)
    return ''.join(out)

for _ in range(8): print(generar(modelo))

E=modelo.embedding.weight.detach().cpu(); C=E-E.mean(0,keepdim=True)
U,S,Vh=torch.linalg.svd(C,full_matrices=False)
coords=... # TODO 16: proyección 2D
plt.figure(figsize=(8,6)); plt.scatter(coords[:,0],coords[:,1])
for i,t in enumerate(vocabulario): plt.annotate('␠' if t==' ' else t,(coords[i,0],coords[i,1]))
plt.grid(alpha=.2); plt.show()

## 8. Contexto y límite arquitectónico
Repite el entrenamiento para $m\in\{1,2,4,8\}$. Cuenta parámetros y compara validación. Distingue: (i) información fuera de la ventana, (ii) capacidad de representación, (iii) dificultad de optimización.

In [ ]:
def nparams(m):
    return sum(p.numel() for p in m.parameters())

resultados=[]
for mm in [1,2,4,8]:
    Xt,yt=crear_ejemplos(corpus,mm)
    p=torch.randperm(len(Xt)); c=int(.8*len(Xt)); a,b=p[:c],p[c:]
    mod=ModeloVentana(V,12,mm,48).to(device)
    # TODO 17: entrena y evalúa
    ...
print(resultados)

# Problema final abierto — ¿Cuánto contexto necesita realmente la red?
Construye secuencias $A r_1\cdots r_L A$ o $B r_1\cdots r_L B$, con $r_i\in\{x,y,z\}$ i.i.d. y primer símbolo equiprobable. El objetivo es predecir **solo el último símbolo**.

Para $L=6$ compara $m=3$ con $m=7$.

**Entrega:** 1) al menos 2000 ejemplos y split train/validation; 2) demuestra antes de entrenar que para $m\le L$ el contexto final es independiente del objetivo y la mejor exactitud media es $1/2$; 3) entrena ambos modelos; 4) reporta CE, perplexity y accuracy; 5) cuenta parámetros y razona por qué el salto de rendimiento no se explica solo por tener más parámetros; 6) comprueba qué filas del embedding reciben gradiente; 7) repite con otro $L$; 8) explica la diferencia entre acceso a información, capacidad y optimización; 9) propone cómo una RNN y una conexión directa tipo atención cambiarían el problema.

In [ ]:
# RETO A — datos
L=6; N=2400
...
# RETO B — contextos finales m=3 y m=L+1
...
# RETO C — entrenamiento y comparación
...
# RETO D — gradientes y otro L
...